In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')
LOG_DIR = "/content/drive/MyDrive/Research_Logs"

print(f"✅ ログ保存先: {LOG_DIR}")

import huggingface_hub
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
huggingface_hub.login(hf_token)

%pip install rank_bm25
%pip install xopen

from pathlib import Path
import sys
import torch
from datasets import load_dataset
from datetime import datetime
import xopen
import random

random.seed(0)


# 以下自作モジュール
module_path = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/my_modules"
if module_path not in sys.path:
    sys.path.append(module_path)

from ScalingController import ScalingController
from get_model_safe_last_attnweight import get_model_safe, test_patched_model_sanity, remove_cache
from main_inference import main_inference
from run_eval import run_eval_exact_match
from analyze_log import ExperimentAnalyzer

# if 'CACHED_MODEL' not in globals():
#     CACHED_MODEL = None
#     CACHED_TOKENIZER = None

def confirmation():
    """実行前にユーザーの確認を求める"""
    print("\n" + "!" * 30)
    print("警告: 書き出しモードが 'a' (追記) です。")
    print("既存のファイルにデータが追加されますが、よろしいですか？")
    print("!" * 30 + "\n")

    answer = input("実行する場合は 'yes' と入力してください: ").strip().lower()

    if answer != 'yes':
        print("\n[中止] 実行がキャンセルされました。")
        # Jupyterでセルを止める最もクリーンな方法
        raise KeyboardInterrupt("User cancelled the execution.")

    print("\n[開始] 実験を実行します...\n")

Mounted at /content/drive
✅ ログ保存先: /content/drive/MyDrive/Research_Logs
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.4/132.4 kB 19.3 MB/s eta 0:00:00


In [4]:
from datasets.formatting.formatting import TypeVar
from typing import List, Optional, Type
from pydantic.dataclasses import dataclass

import json
import torch
import gc
import os
from tqdm import tqdm
from xopen import xopen
from copy import deepcopy
from itertools import islice
from typing import List, Dict
from transformers import AutoTokenizer, AutoModelForCausalLM, LlamaTokenizer


PROMPTS_ROOT = Path("/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/lost_in_the_middle/prompts")
T = TypeVar("T")

@dataclass(frozen=True)
class Document:
    title: str
    text: str
    id: Optional[str] = None
    score: Optional[float] = None
    hasanswer: Optional[bool] = None
    isgold: Optional[bool] = None
    original_retrieval_index: Optional[int] = None

    @classmethod
    def from_dict(cls: Type[T], data: dict) -> T:
        data = deepcopy(data)
        if not data:
            raise ValueError("Must provide data for creation of Document from dict.")
        id = data.pop("id", None)
        score = data.pop("score", None)
        isgold = data.pop("isgold", None)
        # Convert score to float if it's provided.
        if score is not None:
            score = float(score)
        return cls(**dict(data, id=id, score=score, isgold=isgold))

@dataclass
class Document:
    title: str
    text: str
    @classmethod
    def from_dict(cls, data):
        return cls(title=data.get("title", ""), text=data.get("text", ""))

# --- 再配置ロジック ---
def reorder_docs_by_attention(documents: List[Document], attnmap_data: List, target_trials: List[int]):
    """
    指定された試行のアテンション総和を計算し、降順で文書を並び替える
    """
    # attnmap: [Trial(20), Doc(20), Layer(6), Head(32)]
    attn_array = torch.tensor(attnmap_data)

    # 1. 特定の試行のみを抽出
    if target_trials is not None:
      attn_array = attn_array[target_trials, :, :, :]

    # 2. 全軸（試行, レイヤー, ヘッド）で合計をとる -> (Doc: 20,)
    # axis=0: Trial, axis=2: Layer, axis=3: Head
    doc_scores = attn_array.sum(dim=(0, 2, 3)).tolist()

    # 3. スコアと文書をペアにして降順（高い順）にソート
    doc_score_pairs = sorted(zip(doc_scores, documents), key=lambda x: x[0], reverse=True)

    return [doc for score, doc in doc_score_pairs]

# --- 推論・プロンプト関数 ---
def get_qa_prompt(question: str, documents: List[Document]):
    # FORMAT: Document [1](Title: ...) ...
    formatted_documents = []
    for i, doc in enumerate(documents):
        formatted_documents.append(f"Document [{i+1}](Title: {doc.title}) {doc.text}")

    # 標準的なQAプロンプトテンプレート
    prompt_template = "Answer the question based on the following documents.\n\n{search_results}\n\nQuestion: {question}\nAnswer:"
    return prompt_template.format(question=question, search_results="\n".join(formatted_documents))

def run_batch_inference(prompts: List[str], model, tokenizer):
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    input_len = inputs.input_ids.shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.0,
            do_sample=False,
            use_cache=True
        )

    # バッチ全体の生成テキストをデコード
    results = []
    for i in range(len(prompts)):
        generated_tokens = outputs[i][input_len:]
        text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        results.append(text.strip())
    return results


In [6]:
# --- メイン処理 ---
VALIDATION_NUM = 600

def main():
    # モデルの準備
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.float16,
        use_cache=True
    )

    done_count = 0
    if os.path.exists(OUTPUT_PATH):
        with open(OUTPUT_PATH, "r") as f:
            done_count = sum(1 for _ in f)

    total_skip = VALIDATION_NUM + done_count
    attn_skip = done_count

    # ファイルを同時に読み込み
    with xopen(NQ_path, "r") as f_data, xopen(ATTN_PATH, "r") as f_attn, xopen(OUTPUT_PATH, "a") as f_out:
        f_data_skipped = islice(f_data, total_skip, None)
        f_attn_skipped = islice(f_attn, attn_skip, None)

        pbar = tqdm(desc="Processing Batches")

        while True:
            # BATCH_SIZE分だけ各ファイルから読み込み
            lines_data = list(islice(f_data_skipped, BATCH_SIZE))
            lines_attn = list(islice(f_attn_skipped, BATCH_SIZE))

            if not lines_data or not lines_attn:
                break

            batch_prompts = []
            batch_original_data = []

            for l_data, l_attn in zip(lines_data, lines_attn):
                data = json.loads(l_data)
                attn_json = json.loads(l_attn)

                # 文書リスト作成
                docs = [Document.from_dict(ctx) for ctx in data["ctxs"]]

                # アテンション重みに基づいて再配置
                reordered_docs = reorder_docs_by_attention(docs, attn_json["attnmap"], TARGET_TRIALS)

                # プロンプト作成
                prompt = get_qa_prompt(data["question"], reordered_docs)

                batch_prompts.append(prompt)
                batch_original_data.append(data)

            # バッチ推論
            predictions = run_batch_inference(batch_prompts, model, tokenizer)

            # 結果の書き出し
            for original_data, pred in zip(batch_original_data, predictions):
                output_obj = deepcopy(original_data)
                output_obj["prediction"] = pred
                # どの順番に並べ替えたかのヒントとしてスコアを記録したい場合はここに追加可能
                f_out.write(json.dumps(output_obj) + "\n")

            f_out.flush()
            os.fsync(f_out.fileno())
            pbar.update(len(lines_data))


        del model, tokenizer
        gc.collect()
        torch.cuda.empty_cache()

    print(f"\nProcessing complete. Results saved to: {OUTPUT_PATH}")

In [ ]:
# --- shift ---
MODEL = "llama"
MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"
REORDER_MODE = "shift_bm25"
TRIAL = 12
BATCH_SIZE = 8
TARGET_TRIALS = [0, 2, 4, 6, 8, 9, 10, 12, 14, 16, 18, 19]

NQ_path = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
ATTN_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_{REORDER_MODE}_{MODEL}.jsonl"
OUTPUT_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_{REORDER_MODE}_{TRIAL}trial_{MODEL}.jsonl"


if __name__ == "__main__":
    main()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Processing Batches: 0it [00:03, ?it/s]


Processing complete. Results saved to: /content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_shift_bm25_12trial_llama.jsonl


In [ ]:
# --- shift ---
MODEL = "llama"
MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"
REORDER_MODE = "shift_bm25"
TRIAL = 6
BATCH_SIZE = 8
TARGET_TRIALS = [0, 4, 8, 12, 16, 19]

NQ_path = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
ATTN_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_{REORDER_MODE}_{MODEL}.jsonl"
OUTPUT_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_{REORDER_MODE}_{TRIAL}trial_{MODEL}.jsonl"

if __name__ == "__main__":
    main()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Processing Batches: 0it [00:04, ?it/s]


Processing complete. Results saved to: /content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_shift_bm25_6trial_llama.jsonl


In [ ]:
# --- random ---
MODEL = "llama"
MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"
REORDER_MODE = "random"
TRIAL_NUM = 12
BATCH_SIZE = 8
TARGET_TRIALS = None

NQ_path = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
ATTN_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_{REORDER_MODE}_{TRIAL_NUM}trial_{MODEL}.jsonl"
OUTPUT_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_{REORDER_MODE}_{TRIAL_NUM}trial_{MODEL}.jsonl"


if __name__ == "__main__":
    main()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Processing Batches: 0it [00:02, ?it/s]


Processing complete. Results saved to: /content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_random_12trial_llama.jsonl


In [ ]:
# --- random ---
MODEL = "llama"
MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"
REORDER_MODE = "random"
TRIAL_NUM = 6
BATCH_SIZE = 8
TARGET_TRIALS = None

NQ_path = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
ATTN_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_{REORDER_MODE}_{TRIAL_NUM}trial_{MODEL}.jsonl"
OUTPUT_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_{REORDER_MODE}_{TRIAL_NUM}trial_{MODEL}.jsonl"

if __name__ == "__main__":
    main()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Processing Batches: 0it [00:01, ?it/s]


Processing complete. Results saved to: /content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_random_6trial_llama.jsonl


In [ ]:
# --- block_random ---
MODEL = "llama"
MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"
REORDER_MODE = "block_random"
TRIAL_NUM = 12
BATCH_SIZE = 8
TARGET_TRIALS = None

NQ_path = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
ATTN_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_{REORDER_MODE}_{TRIAL_NUM}trial_{MODEL}.jsonl"
OUTPUT_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_{REORDER_MODE}_{TRIAL_NUM}trial_{MODEL}.jsonl"

if __name__ == "__main__":
    main()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Processing Batches: 0it [00:02, ?it/s]


Processing complete. Results saved to: /content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_block_random_12trial_llama.jsonl


In [ ]:
# --- block_random ---
MODEL = "llama"
MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"
REORDER_MODE = "block_random"
TRIAL_NUM = 6
BATCH_SIZE = 8
TARGET_TRIALS = None

NQ_path = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
ATTN_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_{REORDER_MODE}_{TRIAL_NUM}trial_{MODEL}.jsonl"
OUTPUT_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_{REORDER_MODE}_{TRIAL_NUM}trial_{MODEL}.jsonl"

if __name__ == "__main__":
    main()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Processing Batches: 0it [00:01, ?it/s]


Processing complete. Results saved to: /content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_block_random_6trial_llama.jsonl


In [7]:
from transformers import LlamaTokenizer
# --- メイン処理 ---
VALIDATION_NUM = 600

def main():
    # モデルの準備
    tokenizer = LlamaTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.float16,
        use_cache=True
    )

    done_count = 0
    if os.path.exists(OUTPUT_PATH):
        with open(OUTPUT_PATH, "r") as f:
            done_count = sum(1 for _ in f)

    total_skip = VALIDATION_NUM + done_count
    attn_skip = done_count

    # ファイルを同時に読み込み
    with xopen(NQ_path, "r") as f_data, xopen(ATTN_PATH, "r") as f_attn, xopen(OUTPUT_PATH, "a") as f_out:
        f_data_skipped = islice(f_data, total_skip, None)
        f_attn_skipped = islice(f_attn, attn_skip, None)

        pbar = tqdm(desc="Processing Batches")

        while True:
            # BATCH_SIZE分だけ各ファイルから読み込み
            lines_data = list(islice(f_data_skipped, BATCH_SIZE))
            lines_attn = list(islice(f_attn_skipped, BATCH_SIZE))

            if not lines_data or not lines_attn:
                break

            batch_prompts = []
            batch_original_data = []

            for l_data, l_attn in zip(lines_data, lines_attn):
                data = json.loads(l_data)
                attn_json = json.loads(l_attn)

                # 文書リスト作成
                docs = [Document.from_dict(ctx) for ctx in data["ctxs"]]

                # アテンション重みに基づいて再配置
                reordered_docs = reorder_docs_by_attention(docs, attn_json["attnmap"], TARGET_TRIALS)

                # プロンプト作成
                prompt = get_qa_prompt(data["question"], reordered_docs)

                batch_prompts.append(prompt)
                batch_original_data.append(data)

            # バッチ推論
            predictions = run_batch_inference(batch_prompts, model, tokenizer)

            # 結果の書き出し
            for original_data, pred in zip(batch_original_data, predictions):
                output_obj = deepcopy(original_data)
                output_obj["prediction"] = pred
                # どの順番に並べ替えたかのヒントとしてスコアを記録したい場合はここに追加可能
                f_out.write(json.dumps(output_obj) + "\n")

            f_out.flush()
            pbar.update(len(lines_data))


        del model, tokenizer
        gc.collect()
        torch.cuda.empty_cache()

    print(f"\nProcessing complete. Results saved to: {OUTPUT_PATH}")

In [8]:
# --- shift ---
MODEL = "vicuna-16k"
MODEL_NAME = "lmsys/vicuna-7b-v1.5-16k"
REORDER_MODE = "shift_bm25"
TRIAL = 12
BATCH_SIZE = 16
TARGET_TRIALS = [0, 2, 4, 6, 8, 9, 10, 12, 14, 16, 18, 19]

NQ_path = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
ATTN_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_{REORDER_MODE}_{MODEL}.jsonl"
OUTPUT_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_{REORDER_MODE}_{TRIAL}trial_{MODEL}.jsonl"


if __name__ == "__main__":
    main()

tokenizer_config.json:   0%|          | 0.00/750 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/692 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Processing Batches: 192it [03:23,  1.04s/it]This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (4096). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.
Processing Batches: 2055it [36:14,  1.06s/it]


Processing complete. Results saved to: /content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_shift_bm25_12trial_vicuna-16k.jsonl


In [9]:
# --- shift ---
MODEL = "vicuna-16k"
MODEL_NAME = "lmsys/vicuna-7b-v1.5-16k"
REORDER_MODE = "shift_bm25"
TRIAL = 6
BATCH_SIZE = 8
TARGET_TRIALS = [0, 4, 8, 12, 16, 19]

NQ_path = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
ATTN_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_{REORDER_MODE}_{MODEL}.jsonl"
OUTPUT_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_{REORDER_MODE}_{TRIAL}trial_{MODEL}.jsonl"

if __name__ == "__main__":
    main()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Processing Batches: 2055it [38:51,  1.13s/it]


Processing complete. Results saved to: /content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_shift_bm25_6trial_vicuna-16k.jsonl


In [10]:
# --- random ---
MODEL = "vicuna-16k"
MODEL_NAME = "lmsys/vicuna-7b-v1.5-16k"
REORDER_MODE = "random"
TRIAL_NUM = 12
BATCH_SIZE = 8
TARGET_TRIALS = None

NQ_path = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
ATTN_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_{REORDER_MODE}_{TRIAL_NUM}trial_{MODEL}.jsonl"
OUTPUT_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_{REORDER_MODE}_{TRIAL_NUM}trial_{MODEL}.jsonl"


if __name__ == "__main__":
    main()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Processing Batches: 2055it [38:13,  1.12s/it]


Processing complete. Results saved to: /content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_random_12trial_vicuna-16k.jsonl


In [11]:
# --- random ---
MODEL = "vicuna-16k"
MODEL_NAME = "lmsys/vicuna-7b-v1.5-16k"
REORDER_MODE = "random"
TRIAL_NUM = 6
BATCH_SIZE = 8
TARGET_TRIALS = None

NQ_path = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
ATTN_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_{REORDER_MODE}_{TRIAL_NUM}trial_{MODEL}.jsonl"
OUTPUT_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_{REORDER_MODE}_{TRIAL_NUM}trial_{MODEL}.jsonl"

if __name__ == "__main__":
    main()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Processing Batches: 2055it [37:52,  1.11s/it]


Processing complete. Results saved to: /content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_random_6trial_vicuna-16k.jsonl


In [12]:
# --- block_random ---
MODEL = "vicuna-16k"
MODEL_NAME = "lmsys/vicuna-7b-v1.5-16k"
REORDER_MODE = "block_random"
TRIAL_NUM = 12
BATCH_SIZE = 8
TARGET_TRIALS = None

NQ_path = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
ATTN_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_{REORDER_MODE}_{TRIAL_NUM}trial_{MODEL}.jsonl"
OUTPUT_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_{REORDER_MODE}_{TRIAL_NUM}trial_{MODEL}.jsonl"

if __name__ == "__main__":
    main()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Processing Batches: 2055it [38:05,  1.11s/it]


Processing complete. Results saved to: /content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_block_random_12trial_vicuna-16k.jsonl


In [13]:
# --- block_random ---
MODEL = "vicuna-16k"
MODEL_NAME = "lmsys/vicuna-7b-v1.5-16k"
REORDER_MODE = "block_random"
TRIAL_NUM = 6
BATCH_SIZE = 8
TARGET_TRIALS = None

NQ_path = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
ATTN_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_{REORDER_MODE}_{TRIAL_NUM}trial_{MODEL}.jsonl"
OUTPUT_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_{REORDER_MODE}_{TRIAL_NUM}trial_{MODEL}.jsonl"

if __name__ == "__main__":
    main()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Processing Batches: 2055it [37:16,  1.09s/it]


Processing complete. Results saved to: /content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_output/reordered_generation_results_block_random_6trial_vicuna-16k.jsonl


In [ ]:
import time
import gc
import torch
from google.colab import runtime

# 1. メモリの最終開放（念のため）
gc.collect()
torch.cuda.empty_cache()

# 2. 待機設定
# Google Driveの同期には30〜60秒ほど見ておくと非常に安全です
WAIT_TIME_SEC = 300

print(f"✅ 全ての処理が完了しました。")
print(f"📂 Google Driveへの反映を待機しています（{WAIT_TIME_SEC}秒）...")

# プログレスバー付きで待機（あとどれくらいで消えるか視覚化）
from tqdm import tqdm
for _ in tqdm(range(WAIT_TIME_SEC)):
    time.sleep(1)

print("\n🚀 待機完了。セッションを終了し、ランタイムを削除します。お疲れ様でした！")

# 3. セッション終了
runtime.unassign()